# Modeling-knob screen for the injection (bias) study

Which GENIE (`*_UBGenie`) and second-class-current (`*_SCC`) splines move the fitted
log10(Q^2) x p_n spectrum enough to be worth an injection fit? For every knob this notebook
builds the per-bin ratio PROfit itself uses, `N(knob = k) / N(knob = 0)` in each reco bin
(`force_0_cv`, weights restricted by the knob's `include_only_weights`, NaN/inf and > 30 weights
set to 1 as in `PROcreate`), in the binning of the chosen binning-study variant, and keeps every knob whose +-1 sigma shift
exceeds `IGNORE_BELOW` in some bin.

The knobs it selects are written to `xml/injection_study/knobs.json`, which
`python/scripts/make_injection_study_xmls.py` reads to write one XML per (base fit, knob): the
base fit with only that knob turned from `spline_to_covariance` into a `spline`, so it can be
injected with `--inject-systs` and held at its CV with `--fix`, and every other modeling knob
(including the `RPA_CCQE` and `NormCCMEC` splines the M_A fits float) in the covariance. `xml/injection_study/run_all.sh`
runs them, and Section 8 reads their results back: the bias of the fitted axial physics per knob.

Flux, hadron-production (`piprod`), re-interaction, POT normalization and detector systematics are
not screened. The axial knobs (`MaCCQE_UBGenie`, `AxFFCCQEshape_UBGenie`) are the fitted
physics, not modeling knobs, and are excluded.

Knob-to-vector-entry mapping follows the XML `knobvals` exactly as PROfit reads it; Section 6
records which entry equals `TunedCentralValue_UBGenie` for reference only.

In [ ]:
from datetime import date
import json
import re
from pathlib import Path

import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot
from IPython.display import display

start = Path.cwd().resolve()
REPO = next(
    (parent / 'ma_zexp' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'xml' / 'binning_study').is_dir()),
    None,
)
if REPO is None:
    raise FileNotFoundError('Could not locate ma_zexp/xml/binning_study')

## Controls

In [ ]:
VARIANT = 'count50'                         # binning-study variant whose XMLs are the base fits
BASE_FITS = ['ma', 'lqcd_k6']             # one dipole M_A fit and one z-expansion fit (the 2026-10-01 run used the former 'ma_no_axff', whose results stay under STUDY_ROOT/ma_no_axff)
BASE_DIR = REPO / 'xml' / 'binning_study' / 'asimov' / VARIANT

EXCLUDED_KNOBS = {'MaCCQE_UBGenie', 'AxFFCCQEshape_UBGenie'}   # the fitted axial physics
KNOB_PATTERN = re.compile(r'\w+_UBGenie|\w+_SCC')

IGNORE_BELOW = 0.01     # max |shift| over bins below this: not worth injecting; everything above is kept

SIGMAS = [1.0, 3.0]          # injected shifts for knobs with knots at -3..3
TWO_POINT_SIGMAS = [1.0]     # knobvals "0, 1" or "1, 0": 1 is the full alternative model

OUTPUT_JSON = REPO / 'xml' / 'injection_study' / 'knobs.json'
CACHE = REPO / 'tables' / 'injection_study' / 'selected_events.npz'
FIG_DIR = REPO / 'figs' / 'injection_study'
RECOMPUTE = False       # re-read the ROOT file even if the cache exists

## Section 1: the base fits

Read the fitted binning, the POT scale and the systematics block of every base XML. The knob list
is the union of the uncommented `*_UBGenie` / `*_SCC` systematics, with each knob's type,
`knobvals` and `include_only_weights` per base fit.

In [ ]:
def strip_comments(text):
    return re.sub(r'<!--.*?-->', '', text, flags=re.S)


def attr(tag, name, default=None):
    m = re.search(rf'\b{name}="([^"]*)"', tag)
    return m.group(1) if m else default


def parse_base(path):
    text = strip_comments(path.read_text())
    reco = re.search(r'<bins2D\s[^>]*unit="log10\(Q\^2 / GeV\^2\);p_n"[^>]*/>', text).group(0)
    edges_q2 = np.array(attr(reco, 'edgesx').split(), float)
    edges_pn = np.array(attr(reco, 'edgesy').split(), float)
    det_pot = float(re.search(r'<detector\s[^>]*pot="([^"]+)"', text).group(1))
    mcfile = re.search(r'<MCFile\s[^>]*>', text).group(0)
    mc_pot = float(attr(mcfile, 'pot'))
    branch = re.search(r'<branch\s(.*?)>', text, flags=re.S).group(1)
    weights = dict(re.findall(r'weight_(\d+)\s*=\s*"([^"]*)"', branch))
    assert '<variable2D>log10(afro_1mu1p_Q2);afro_1mu1p_Pn</variable2D>' in text
    all_knobs = {}
    for tag, name in re.findall(r'<systematic\s([^>]*)>\s*([^<\s]+)\s*</systematic>', text):
        if not KNOB_PATTERN.fullmatch(name):
            continue
        kv = attr(tag, 'knobvals')
        include = attr(tag, 'include_only_weights')
        all_knobs[name] = dict(
            type=attr(tag, 'type'), plotname=attr(tag, 'plotname'), tag=attr(tag, 'tag'),
            knobvals=[float(v) for v in kv.split(',')] if kv else [-3., -2., -1., 0., 1., 2., 3.],
            include=[int(i) for i in include.split(',')] if include else None,
        )
    knobs = {k: v for k, v in all_knobs.items() if k not in EXCLUDED_KNOBS}   # the screened ones
    return dict(edges_q2=edges_q2, edges_pn=edges_pn, pot_scale=det_pot / mc_pot,
                weights=weights, knobs=knobs, all_knobs=all_knobs)


BASES = {fit: parse_base(BASE_DIR / f'{fit}.xml') for fit in BASE_FITS}
first = BASES[BASE_FITS[0]]
EDGES_Q2, EDGES_PN = first['edges_q2'], first['edges_pn']
for fit, b in BASES.items():
    assert np.array_equal(b['edges_q2'], EDGES_Q2) and np.array_equal(b['edges_pn'], EDGES_PN), fit
NQ2, NPN = len(EDGES_Q2) - 1, len(EDGES_PN) - 1
KNOBS = sorted(set().union(*(b['knobs'] for b in BASES.values())))

print(f'{VARIANT}: {NQ2} log10(Q^2) x {NPN} p_n = {NQ2 * NPN} bins, {len(KNOBS)} knobs screened')
for fit, b in BASES.items():
    extra = {k: v for k, v in b['weights'].items() if k != '1'}
    print(f'  {fit}: POT scale {b["pot_scale"]:.3f}, CV weights {extra}')

knob_table = pd.DataFrame({
    (fit, col): {k: (str(b['knobs'][k][col]) if k in b['knobs'] else '-') for k in KNOBS}
    for fit, b in BASES.items() for col in ('type', 'include')
})
knob_table.insert(0, 'knobvals', [str(first['knobs'].get(k, {}).get('knobvals')) for k in KNOBS])
display(knob_table)

## Section 2: selected MC events

Selected overlay MC only (`afro_1mu1p_sel==1`, no data, EXT, dirt or NuWro), both subchannels.
Every knob's weight vector and every weight branch the base fits use are cached to
`tables/injection_study/selected_events.npz`, so the rest of the notebook does not touch the
14 GB input file. Delete the cache or set `RECOMPUTE = True` after the input file changes.

In [ ]:
INPUT_FILE = re.search(r'<MCFile\s[^>]*filename="([^"]+)"', (BASE_DIR / f'{BASE_FITS[0]}.xml').read_text()).group(1)
# weight_N attributes are TTreeFormula expressions (e.g. the M_A split net_weight/(non_genie_net_weight + ...));
# the ones used here are plain arithmetic on branches, which numpy evaluates the same way.
weight_exprs = sorted({w for b in BASES.values() for k, w in b['weights'].items() if k != '1'})
assert all(re.fullmatch(r'[\w\s()+\-*/.=<>]+', w) and '&&' not in w for w in weight_exprs), weight_exprs
weight_branches = sorted({t for w in weight_exprs for t in re.findall(r'[A-Za-z_]\w*', w)})
BASE_BRANCHES = ['afro_1mu1p_Q2', 'afro_1mu1p_Pn', 'afro_1mu1p_true', 'TunedCentralValue_UBGenie']
CUT_BRANCHES = ['isdata', 'isext', 'isdirt', 'isnuwro', 'afro_1mu1p_sel']


def read_selected(names):
    tree = uproot.open(INPUT_FILE)['tree']
    parts = {}
    for a in tree.iterate(CUT_BRANCHES + names, step_size=150_000):
        a = a[(a.isdata == 0) & (a.isext == 0) & (a.isdirt == 0) & (a.isnuwro == 0) & (a.afro_1mu1p_sel == 1)]
        for name in names:
            col = a[name]
            parts.setdefault(name, []).append(
                ak.to_numpy(col) if col.ndim == 1 else ak.to_numpy(ak.to_regular(col, axis=1)))
    return {name: np.concatenate(p) for name, p in parts.items()}


AXIAL_IN_BASES = sorted({k for b in BASES.values() for k in b['all_knobs']} - set(KNOBS))   # the fitted axial splines
NEEDED = BASE_BRANCHES + weight_branches + KNOBS + AXIAL_IN_BASES
stat = Path(INPUT_FILE).stat()
INPUT_STAMP = f'{INPUT_FILE} size={stat.st_size} mtime={int(stat.st_mtime)}'   # the cache is tied to this file version
if not RECOMPUTE and CACHE.exists():
    with np.load(CACHE) as f:
        if str(f['input_file']) != INPUT_STAMP:
            print('input file changed since the cache was written: re-reading')
            RECOMPUTE = True
if RECOMPUTE or not CACHE.exists():
    EV = read_selected(NEEDED)
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    np.savez(CACHE, input_file=INPUT_STAMP, **EV)
else:
    with np.load(CACHE) as f:
        EV = {k: f[k] for k in f.files if k != 'input_file'}
    missing = [k for k in NEEDED if k not in EV]
    if missing:
        # same selection and event order, so only the new branches are read
        print(f'adding {missing} to the cache')
        EV.update(read_selected(missing))
        np.savez(CACHE, input_file=INPUT_STAMP, **EV)

with np.errstate(divide='ignore', invalid='ignore'):
    iq = np.digitize(np.log10(EV['afro_1mu1p_Q2']), EDGES_Q2) - 1
ip = np.digitize(EV['afro_1mu1p_Pn'], EDGES_PN) - 1
IN_RANGE = (iq >= 0) & (iq < NQ2) & (ip >= 0) & (ip < NPN)
BIN = np.where(IN_RANGE, iq * NPN + ip, -1)          # x-major flat order
SIGNAL = EV['afro_1mu1p_true'] == 1
NBINS = NQ2 * NPN


def weight_array(expr):
    return np.asarray(eval(expr, {'__builtins__': {}}, EV), dtype=float)


def hist(w, mask=None):
    m = IN_RANGE if mask is None else IN_RANGE & mask
    return np.bincount(BIN[m], weights=None if w is None else w[m], minlength=NBINS)


RAW = hist(None)
print(f'{len(BIN)} selected events, {IN_RANGE.sum()} inside the binning; '
      f'raw MC per bin: min {RAW.min():.0f}, median {np.median(RAW):.0f}')

## Section 3: per-bin knob ratios

For base fit `b` and knob `k` the universe weight of an event is the product of its
`include_only_weights` (weight_1 is the selection cut, so `"1"` means the bare knob weight and
`"1,2"` multiplies in weight_2) times the knob's vector entry, and the spline ratio in bin `i` is

    r_k(i) = sum_events(included * v_k) / sum_events(included * v_0)

with `v_0` the entry at knob value 0 (`force_0_cv`). The sums are taken at the knots the XML
`knobvals` assign to the stored entries and interpolated linearly in the knob value where a
shift falls between knots; this matters for `RPA_CCQE_UBGenie`, whose knots sit at -3.6225,
..., 2.3775 so that knob 0 is the tuned CV (see notebook 20). The vectors are linear in the
knob, so the interpolation is exact. The CV prediction `N(i)`, used only for the
statistics-only chi^2 below, is the product of all the branch weights times the POT scale.

In [ ]:
def clip(v):
    # PROcreate: NaN/inf and weights above 30 are set to 1
    return np.where(~np.isfinite(v) | (v > 30), 1.0, v)


def included_weight(base, spec):
    w = np.ones(len(BIN))
    for i in (spec['include'] or []):
        if i != 1:
            w = w * weight_array(base['weights'][str(i)])
    return w


def cv_prediction(base, mask=None):
    w = np.full(len(BIN), base['pot_scale'])
    for i, expr in base['weights'].items():
        if i != '1':
            w = w * weight_array(expr)
    return hist(w, mask)


def knot_sums(fit, knob, mask=None):
    '''(sorted knots, per-knot bin sums of included weight x vector entry) for the knob as the base fit configures it.'''
    base = BASES[fit]
    spec = base['knobs'][knob]
    v = clip(EV[knob])
    kv = spec['knobvals']
    if v.shape[1] != len(kv):
        raise ValueError(f'{knob}: {v.shape[1]} vector entries, {len(kv)} knobvals')
    inc = included_weight(base, spec) if spec['include'] else np.ones(len(BIN))
    order = np.argsort(kv)
    return np.array(kv)[order], np.array([hist(inc * v[:, j], mask) for j in order])


def at_knob(knots, sums, value):
    '''Bin sums at a knob value: a knot's own sums, or linear interpolation (extrapolation) between knots.'''
    j = np.searchsorted(knots, value)
    if j < len(knots) and knots[j] == value:
        return sums[j]
    j = min(max(j, 1), len(knots) - 1)
    t = (value - knots[j - 1]) / (knots[j] - knots[j - 1])
    return sums[j - 1] + t * (sums[j] - sums[j - 1])


def shift_values(fit, knob):
    '''The knob values the screen looks at: 1 for a two-point knob, -3..3 for a seven-point one.'''
    kv = BASES[fit]['knobs'][knob]['knobvals']
    return [1.0] if len(kv) == 2 else [-3.0, -2.0, -1.0, 1.0, 2.0, 3.0]


def knob_ratios(fit, knob):
    # {knob value: per-bin ratio to the sums at knob 0} at the standard shifts
    knots, sums = knot_sums(fit, knob)
    den = at_knob(knots, sums, 0.0)
    with np.errstate(divide='ignore', invalid='ignore'):
        return {k: np.where(den > 0, at_knob(knots, sums, k) / den, 1.0) for k in shift_values(fit, knob)}


CV = {fit: cv_prediction(BASES[fit]) for fit in BASE_FITS}
CV_SIG = {fit: cv_prediction(BASES[fit], SIGNAL) for fit in BASE_FITS}
RATIOS = {(fit, k): knob_ratios(fit, k) for fit in BASE_FITS for k in KNOBS if k in BASES[fit]['knobs']}
for fit in BASE_FITS:
    print(f'{fit}: CV prediction {CV[fit].sum():.0f} events, empty bins {(CV[fit] <= 0).sum()}')

### Knob vectors with a non-positive CV entry

GENIE reweighting can return a zero or negative tuned-CV weight. The ROOT writer replaces such an
event's tune weight by 1 in `net_weight` but stores the knob vectors as they come, and PROcreate
only replaces NaN, inf and > 30 entries by 1, so an event with `v_0 <= 0` enters the `force_0_cv`
ratio with a denominator that can cancel against the normal events of a sparse bin. This lists
those events and, per base fit, knob and subchannel (PROfit builds the splines per subchannel),
the bins of this variant whose ratio at some knot exceeds `RATIO_ALARM`, next to the same ratio
with the bad events' vectors set to 1. `MaCCQE_UBGenie`, the fitted axial spline of the M_A
fits, is included.

In [ ]:
RATIO_ALARM = 3.0


def reco_label(i):
    a, b = divmod(int(i), NPN)
    return f'Q2[{EDGES_Q2[a]:.2f},{EDGES_Q2[a + 1]:.2f}] pn[{EDGES_PN[b]:.3g},{EDGES_PN[b + 1]:.3g}]'


T = EV['TunedCentralValue_UBGenie'][:, 0]
BAD_CV = T <= 0
print(f'{BAD_CV.sum()} selected events with a tuned-CV weight <= 0 (T < 0: {(T < 0).sum()}, T == 0: {(T == 0).sum()})')
for i in np.where(T < 0)[0]:
    print(f'  T = {T[i]:.4f}, {"signal" if SIGNAL[i] else "background"}, reco {reco_label(BIN[i]) if BIN[i] >= 0 else "outside the binning"}, '
          f'MaCCQE vector {np.round(EV["MaCCQE_UBGenie"][i], 3).tolist() if "MaCCQE_UBGenie" in EV else "-"}')


def subchannel_ratios(fit, knob, vectors):
    '''{subchannel: (n_knots, NBINS) ratio of the knot sums to the knob-0 sums}, as PROfit builds the spline of each subchannel.'''
    base = BASES[fit]
    spec = base['all_knobs'][knob]
    v = clip(vectors)
    kv = np.array(spec['knobvals'])
    order = np.argsort(kv)
    inc = included_weight(base, spec) if spec['include'] else np.ones(len(BIN))
    out = {}
    for name, m in (('signal', SIGNAL), ('background', ~SIGNAL)):
        sums = np.array([hist(inc * v[:, j], m) for j in order])
        den = at_knob(kv[order], sums, 0.0)
        with np.errstate(divide='ignore', invalid='ignore'):
            out[name] = np.where(den != 0, sums / den, 1.0)
    return out


rows = []
for fit in BASE_FITS:
    for knob in BASES[fit]['all_knobs']:
        cleaned_vectors = EV[knob].copy()
        cleaned_vectors[BAD_CV] = 1.0
        raw, cleaned = subchannel_ratios(fit, knob, EV[knob]), subchannel_ratios(fit, knob, cleaned_vectors)
        for sub, m in (('signal', SIGNAL), ('background', ~SIGNAL)):
            worst = np.abs(raw[sub]).max(axis=0)
            counts = hist(None, m)
            for b in np.where(worst > RATIO_ALARM)[0]:
                rows.append(dict(fit=fit, knob=knob, subchannel=sub, bin=reco_label(b), **{'raw MC': int(counts[b])},
                                 **{'bad events': int(hist(None, m & BAD_CV)[b]), 'max |ratio|': worst[b],
                                    'max |ratio| cleaned': np.abs(cleaned[sub][:, b]).max()}))
ALARMS = pd.DataFrame(rows)
if len(ALARMS):
    print(f'\nbins with a knob ratio above {RATIO_ALARM:g} at some knot:')
    display(ALARMS.style.format({'max |ratio|': '{:.1f}', 'max |ratio| cleaned': '{:.2f}'}))
else:
    print(f'\nno bin has a knob ratio above {RATIO_ALARM:g} at any knot')

## Section 4: size of the +-1 sigma shifts

Per bin the worse of the -1 and +1 sigma shifts (the single alternative for `knobvals="0, 1"`),
then over the bins: the maximum, the 90th percentile, the number of bins above 1% and 2%, the
normalization change at +1 sigma, and the statistics-only chi^2 `sum (delta N)^2 / N` at the
detector POT (the ranking it gives, not its absolute value, is the point: the systematic
covariance is not included). A knob is kept when its maximum over the base fits is at least `IGNORE_BELOW`.

In [ ]:
def bin_label(i):
    a, b = divmod(int(i), NPN)
    return f'Q2[{EDGES_Q2[a]:.2f},{EDGES_Q2[a + 1]:.2f}] pn[{EDGES_PN[b]:.3g},{EDGES_PN[b + 1]:.3g}]'


def one_sigma_shifts(fit, knob):
    r = RATIOS[(fit, knob)]
    return [r[k] - 1 for k in (-1.0, 1.0) if k in r]


def summarize(fit, knob):
    shifts = one_sigma_shifts(fit, knob)
    worst = np.max(np.abs(shifts), axis=0)
    n = CV[fit]
    good = n > 0
    plus = RATIOS[(fit, knob)][1.0]
    ib = int(np.argmax(np.where(good, worst, -1)))
    return {
        'max': worst[good].max(),
        'p90': np.percentile(worst[good], 90),
        'n>1%': int((worst[good] > 0.01).sum()),
        'n>2%': int((worst[good] > 0.02).sum()),
        'norm +1': (plus * n).sum() / n.sum() - 1,
        'chi2 stat': np.mean([((s * n)[good] ** 2 / n[good]).sum() for s in shifts]),
        'worst bin': bin_label(ib),
        'raw MC': int(RAW[ib]),
    }


def classify(row):
    return 'keep' if row['max'] >= IGNORE_BELOW else 'ignore'


rows = []
for knob in KNOBS:
    per_fit = {fit: summarize(fit, knob) for fit in BASE_FITS if (fit, knob) in RATIOS}
    worst_fit = max(per_fit, key=lambda f: per_fit[f]['max'])
    row = dict(per_fit[worst_fit])
    row['max'] = max(s['max'] for s in per_fit.values())
    row['p90'] = max(s['p90'] for s in per_fit.values())
    row.update({f'max {f}': s['max'] for f, s in per_fit.items()})
    row['tag'] = first['knobs'][knob]['tag']
    row['two-point'] = sorted(first['knobs'][knob]['knobvals']) == [0.0, 1.0]
    rows.append(pd.Series(row, name=knob))
SCREEN = pd.DataFrame(rows).sort_values('max', ascending=False)
SCREEN['class'] = SCREEN.apply(classify, axis=1)

pct = ['max', 'p90', 'norm +1'] + [f'max {f}' for f in BASE_FITS]
display(SCREEN.style.format({c: '{:.2%}' for c in pct} | {'chi2 stat': '{:.1f}'}))
print(SCREEN['class'].value_counts().to_string())

The two base fits apply the knobs with different `include_only_weights` (`"1"` in the M_A fit,
`"1,2"` with `non_genie_net_weight` in the z-expansion fits), so their ratios differ slightly. The
largest difference over knobs and bins:

In [ ]:
common = [k for k in KNOBS if all((f, k) in RATIOS for f in BASE_FITS)]
if len(BASE_FITS) > 1:
    f0, f1 = BASE_FITS[:2]
    diff = {k: max(np.max(np.abs(RATIOS[(f0, k)][v] - RATIOS[(f1, k)][v])) for v in RATIOS[(f0, k)]) for k in common}
    worst = max(diff, key=diff.get)
    print(f'{f0} vs {f1}: largest |ratio difference| {diff[worst]:.2e} ({worst})')

## Section 5: where the shifts sit

Maximum and 90th-percentile |shift| per knob against the threshold, then the +1 sigma
fractional shift over the (log10 Q^2, p_n) plane for every knob that will be injected. The colour
scale is symmetric about zero and set per panel, since the knobs differ by an order of magnitude.

In [ ]:
FIG_DIR.mkdir(parents=True, exist_ok=True)
CLASS_COLOR = {'keep': '#2a6fdb', 'ignore': '#dfe2e6'}

fig, ax = plt.subplots(figsize=(7, 0.22 * len(SCREEN) + 1))
y = np.arange(len(SCREEN))[::-1]
ax.barh(y, 100 * SCREEN['max'], height=0.7, color=[CLASS_COLOR[c] for c in SCREEN['class']])
ax.scatter(100 * SCREEN['p90'], y, s=12, color='#1b1f24', zorder=3, label='90th percentile')
ax.axvline(100 * IGNORE_BELOW, color='#1b1f24', lw=0.8, ls=':')
ax.set_yticks(y, SCREEN.index, fontsize=7)
ax.set_xscale('log')
ax.set_xlabel('max over bins of |shift| at 1 sigma [%]')
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in CLASS_COLOR.values()]
ax.legend(handles + [ax.collections[0]], list(CLASS_COLOR) + ['90th percentile'], fontsize=7, loc='lower right')
ax.set_title(f'Knob screen, {VARIANT} binning', fontsize=9)
ax.grid(axis='x', which='both', color='#e5e7eb', lw=0.5)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIG_DIR / f'knob_screen_{VARIANT}.pdf')
plt.show()

In [ ]:
INJECT = [k for k in SCREEN.index if SCREEN.loc[k, 'class'] == 'keep']
fit = BASE_FITS[-1]
ncol = 5
nrow = int(np.ceil(len(INJECT) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 2.5 * nrow), squeeze=False)
for ax, knob in zip(axes.flat, INJECT):
    shift = (RATIOS[(fit, knob)][1.0] - 1).reshape(NQ2, NPN)
    lim = max(np.max(np.abs(shift)), 1e-4) * 100
    mesh = ax.pcolormesh(EDGES_Q2, EDGES_PN, 100 * shift.T, cmap='RdBu_r', vmin=-lim, vmax=lim)
    fig.colorbar(mesh, ax=ax, label='shift [%]', pad=0.02)
    ax.set_title(knob.replace('_UBGenie', ''), fontsize=8)
    ax.tick_params(labelsize=7)
for ax in axes.flat[len(INJECT):]:
    ax.set_visible(False)
for ax in axes[-1]:
    ax.set_xlabel('reco log10(Q^2 / GeV^2)', fontsize=7)
for ax in axes[:, 0]:
    ax.set_ylabel('reco p_n [GeV]', fontsize=7)
fig.suptitle(f'+1 sigma fractional shift, {fit} weights, {VARIANT} binning', fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / f'knob_shift_maps_{VARIANT}.pdf')
plt.show()

Signal versus background: the +1 sigma shift in events split by subchannel, as a fraction of
the total |delta N|. Knobs acting only through the non-signal subchannel (RES, pion FSI) enter the
fit through the background shape; the signal-side ones compete with F_A directly.

In [ ]:
fit = BASE_FITS[-1]
base = BASES[fit]
share = {}
for knob in INJECT:
    parts = {}
    for name, m in (('signal', SIGNAL), ('background', ~SIGNAL)):
        knots, sums = knot_sums(fit, knob, m)
        s0, s1 = at_knob(knots, sums, 0.0), at_knob(knots, sums, 1.0)
        cv = cv_prediction(base, m)
        with np.errstate(divide='ignore', invalid='ignore'):
            parts[name] = np.abs(np.where(s0 > 0, s1 / s0 - 1, 0) * cv).sum()
    share[knob] = parts['signal'] / (parts['signal'] + parts['background'])
display(pd.Series(share, name='signal share of |delta N| at +1 sigma').to_frame().style.format('{:.0%}'))

## Section 6: which vector entry equals the tuned CV (reference only)

Fraction of selected events whose knob entry equals `TunedCentralValue_UBGenie` (rtol 1e-5). For
knobs with knots at -3..3 the knob-0 entry should match for every event; RPA_CCQE matches
nowhere because its knots are shifted (notebook 20). This does not change
anything above: the ratios follow the XML `knobvals`.

In [ ]:
T = EV['TunedCentralValue_UBGenie'][:, 0]
ref = {}
for knob in KNOBS:
    if not knob.endswith('_UBGenie'):
        continue
    v = EV[knob]
    kv = first['knobs'][knob]['knobvals']
    ref[knob] = {f'{k:+g}': np.isclose(v[:, j], T, rtol=1e-5).mean() for j, k in enumerate(kv)}
display(pd.DataFrame(ref).T.style.format('{:.3f}', na_rep=''))

## Section 7: write the injection list

Writes `xml/injection_study/knobs.json`: the binning variant, the base fits and, for each knob to
inject, its shifts in sigma. Regenerate the XMLs afterwards with
`python/scripts/make_injection_study_xmls.py`.

In [ ]:
payload = {
    '_meta': {
        'generated': date.today().isoformat(),
        'notebook': 'python/notebooks/19_modeling_knob_screen.ipynb',
        'variant': VARIANT,
        'base_fits': BASE_FITS,
        'rule': f'inject every knob with max |1 sigma shift| >= {IGNORE_BELOW:g} in some base fit',
    },
    'knobs': {
        k: {
            'sigmas': TWO_POINT_SIGMAS if SCREEN.loc[k, 'two-point'] else SIGMAS,
            'plotname': first['knobs'][k]['plotname'],
            'max_shift': round(float(SCREEN.loc[k, 'max']), 4),
        }
        for k in INJECT
    },
}
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.write_text(json.dumps(payload, indent=1) + '\n')
n_runs = sum(len(v['sigmas']) for v in payload['knobs'].values()) * len(BASE_FITS)
print(f'wrote {OUTPUT_JSON.relative_to(REPO)}: {len(INJECT)} knobs, {n_runs} injection fits over {len(BASE_FITS)} base fits')

## Section 8: results of the injection fits

`xml/injection_study/run_all.sh` writes one directory per XML below
`STUDY_ROOT/<fit>/<knob>/`: the baseline (`profile` stage, giving the 1 sigma uncertainties of the
axial parameters and their MCMC chain) and one fit per injected shift, in which the Asimov data
has the knob at `+sigma` while the fit holds it at its CV. The injected fits run the `profile`
stage too (the runner's default since 2026-09-30): its scan over the axial parameters catches a
minimizer stuck in a local minimum, which the plain `global` stage does not; either stage's
output is read here. The fitted axial parameters
are read from `<fit>_inj<sigma>_global_fit.txt`, the chi^2 at the minimum from the log, and the
baseline uncertainties from `one_sigma_errs` in `<fit>_baseline_PROfile.root`.

The bias of a knob is the shift of the fitted axial physics when that knob is mis-modelled and
not floated, in units of the baseline uncertainty:

* per parameter, `(theta_inj - theta_base) / sigma_base`, with `sigma_base` the profile 1 sigma on
  the side of the shift;
* on the form factor, `[F_A(Q^2; theta_inj) / F_A(Q^2; theta_base) - 1] / delta_F(Q^2)`, with
  `delta_F = sigma[F_A] / |E[F_A]|` from the baseline chain (the definition of notebook 6),
  quoted at `Q2_REFERENCE` and as the maximum over `Q^2 <= Q2_BIAS_MAX`.

Parameters map to physics as in `postfit_physical_parameters.load_fit`: a MaCCQE pull is
`M_A = 1.1 + 0.1 pull` GeV, a z-expansion pull vector is `a = a_prior + T pull` with `T` the
PCA shift matrix of the prior. The `+3 sigma` fits check linearity: `shift(3) / 3 shift(1)` is 1
for a linear response. Knobs whose F_A bias at `Q2_REFERENCE` exceeds `BIAS_NOTABLE` (a fraction
of `delta_F`) are highlighted; runs that have not finished are listed and skipped, so this section
can be rerun while the study is still running.

Read the M_A numbers together with the check at the end of Section 3: a knob ratio that blows up
in one sparse bin gives the fit a spurious narrow minimum away from zero (found 2026-09-30 in the
`RPA_CCQE +3 sigma` M_A fit, where the minimizer's best fit and its chain disagreed) and
distorts every fit that moves M_A off its CV.

In [ ]:
import sys

if str(REPO / 'python' / 'scripts') not in sys.path:
    sys.path.insert(0, str(REPO / 'python' / 'scripts'))
from postfit_physical_parameters import FA_SOURCE_COLORS, SPECS, _fa_observables, _zexp_transform

STUDY_ROOT = Path('/nevis/hopper/data/epelaez/axial_mass/injection_study_fit_results') / VARIANT
RUNS = REPO / 'xml' / 'injection_study' / 'runs.txt'
TABLE_DIR = REPO / 'tables' / 'injection_study'
Q2 = np.linspace(0.0, 2.0, 201)      # GeV^2, F_A curves
Q2_REFERENCE = 0.50                  # GeV^2, where the headline bias is quoted (notebook 6)
Q2_BIAS_MAX = 1.0                    # GeV^2, upper end of the range the maximum bias is taken over
BIAS_NOTABLE = 0.2                   # fraction of delta_F above which a knob's bias is highlighted
SPEC = {spec.key: spec for spec in SPECS}
FIT_COLOR = {fit: FA_SOURCE_COLORS[fit] for fit in BASE_FITS}


def read_runs():
    rows = []
    for line in RUNS.read_text().splitlines():
        if not line or line.startswith('#'):
            continue
        variant, fit, stem, knob, sigma = line.split()
        if variant == VARIANT and fit in BASE_FITS and knob != '-':
            rows.append(dict(fit=fit, knob=knob, sigma=float(sigma), stem=stem))
    return pd.DataFrame(rows)


def read_global_fit(path):
    return {name: float(value)
            for name, value in re.findall(r'^(\S+)\s*:\s*(\S+)\s*$', path.read_text(), flags=re.M)}


def read_chi2(log):
    m = re.findall(r'Global Best Fit chi\^2: (\S+)', log.read_text(errors='replace'))
    return float(m[-1]) if m else np.nan


def chain_branch(tree, label):
    key = label.lower()
    hits = [b for b in tree.keys() if b.lower().replace('weight_spline_', '').replace('_ubgenie', '') == key]
    if len(hits) != 1:
        raise KeyError(f'{label}: chain branches {tree.keys()}')
    return hits[0]


def read_chain(root, labels):
    chains = [k for k in root.keys() if k.split(';')[0].endswith('_mcmc_chain')]
    assert len(chains) == 1, chains
    tree = root[chains[0]]
    return np.column_stack([tree[chain_branch(tree, label)].array(library='np') for label in labels])


def read_baseline(fit):
    path = STUDY_ROOT / fit / 'baseline' / f'{fit}_baseline_PROfile.root'
    if not path.exists():
        return None
    with uproot.open(path) as root:
        hist = root['global_fit_result']
        labels = list(hist.axis().labels())
        best = hist.values()
        errs = root['one_sigma_errs']
        lo, hi = errs.errors(axis='y', which='low'), errs.errors(axis='y', which='high')
        assert len(lo) == len(labels), (labels, lo)
        chain = read_chain(root, labels)
    return dict(labels=labels, best=best, lo=lo, hi=hi, chain=chain,
                chi2=read_chi2(path.parent / 'profile_baseline.log'))


RUN_TABLE = read_runs()
BASELINE = {fit: read_baseline(fit) for fit in BASE_FITS}
for fit in BASE_FITS:
    b = BASELINE[fit]
    if b is None:
        print(f'{fit}: no baseline yet ({STUDY_ROOT / fit / "baseline"})')
        continue
    print(f'{fit}: axial parameters {b["labels"]}, baseline best fit {np.round(b["best"], 3)}, '
          f'-1 sigma {np.round(b["lo"], 3)}, +1 sigma {np.round(b["hi"], 3)}, chi2 {b["chi2"]:.3g}')


def result_path(row):
    return STUDY_ROOT / row.fit / row.stem / f'{row.fit}_inj{row.sigma:g}_global_fit.txt'


def stage_file(row, suffix):
    '''The injected fit's file for the stage that ran (profile preferred over global).

    `suffix` is formatted with the stage name (`stage`: profile, global) and PROfit's ROOT file
    tag for it (`root`: PROfile, PROglobal), e.g. '{fit}_inj{sigma}_{root}.root' or '{stage}_inj{sigma}.log'.
    '''
    for stage, root in (('profile', 'PROfile'), ('global', 'PROglobal')):
        path = result_path(row).with_name(suffix.format(fit=row.fit, sigma=f'{row.sigma:g}', stage=stage, root=root))
        if path.exists():
            return path
    return path


RUN_TABLE['done'] = [result_path(r).exists() for r in RUN_TABLE.itertuples()]
missing = RUN_TABLE[~RUN_TABLE.done]
print(f'{RUN_TABLE.done.sum()} of {len(RUN_TABLE)} injection fits finished')
if len(missing):
    print('not finished:\n' + missing[['fit', 'knob', 'sigma']].to_string(index=False))

Best-fit axial parameters of every finished injection, the chi^2 at the minimum and the F_A
bias at `Q2_REFERENCE`, both at the minimizer's best fit and as the median and 16-84% interval
of the fit's own MCMC chain. The chain median sits off the minimum by the posterior asymmetry
(about a tenth of `delta_F` for M_A); a gap above `DISAGREE` means the minimizer and the chain
disagree on where the fit went and is flagged.

In [ ]:
def to_physics(fit, pulls):
    '''Physical axial parameters for pull vectors (n, n_axial): M_A [GeV] or the z-expansion a_k.'''
    pulls = np.atleast_2d(np.asarray(pulls, float))
    spec = SPEC[fit]
    if spec.prior is None:
        return 1.1 + 0.1 * pulls[:, :1]
    central, matrix = _zexp_transform(spec.prior)
    return central + pulls @ matrix.T


def fa_curves(fit, pulls):
    '''F_A(Q2) on the Q2 grid, one row per pull vector.'''
    phys = to_physics(fit, pulls)
    spec = SPEC[fit]
    if spec.prior is None:
        return -1.2723 / (1 + Q2[None, :] / phys ** 2) ** 2
    return _fa_observables(phys, spec.prior.t0_gev2, spec.prior.t_cut_gev2, Q2)


def signed_sigma(shift, lo, hi):
    return np.where(shift >= 0, hi, lo)


IREF = int(np.argmin(np.abs(Q2 - Q2_REFERENCE)))
IN_RANGE_Q2 = (Q2 > 0) & (Q2 <= Q2_BIAS_MAX)
BAND = {}       # fit -> (E[F_A], sigma[F_A], delta_F) over Q2 from the baseline chain
FA_BASE = {}    # fit -> F_A at the baseline best fit
for fit in BASE_FITS:
    b = BASELINE[fit]
    if b is None:
        continue
    curves = fa_curves(fit, b['chain'])
    mean, sigma = curves.mean(axis=0), curves.std(axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        BAND[fit] = (mean, sigma, np.where(sigma > 0, sigma / np.abs(mean), np.nan))   # F_A(0) is fixed
    FA_BASE[fit] = fa_curves(fit, b['best'])[0]
    print(f'{fit}: baseline delta_F({Q2_REFERENCE:g} GeV^2) = {100 * BAND[fit][2][IREF]:.2f}%, '
          f'chain std of the pulls {np.round(b["chain"].std(axis=0), 3)}')


def bias_curve(fit, pulls):
    '''Fractional F_A shift over Q2 in units of the baseline delta_F, one row per pull vector.'''
    return (fa_curves(fit, pulls) / FA_BASE[fit] - 1) / BAND[fit][2]


records = []
for row in RUN_TABLE[RUN_TABLE.done].itertuples():
    b = BASELINE[row.fit]
    if b is None:
        continue
    values = read_global_fit(result_path(row))
    pulls = np.array([values[label] for label in b['labels']])
    shift = pulls - b['best']
    with uproot.open(stage_file(row, '{fit}_inj{sigma}_{root}.root')) as root:
        chain = read_chain(root, b['labels'])[::10]
    q16, q50, q84 = np.quantile(bias_curve(row.fit, chain)[:, IREF], [0.16, 0.5, 0.84])
    bias = bias_curve(row.fit, pulls)[0]
    rec = dict(fit=row.fit, knob=row.knob, sigma=row.sigma,
               chi2=read_chi2(stage_file(row, '{stage}_inj{sigma}.log')),
               phys=to_physics(row.fit, pulls)[0], bias=bias,
               **{f'{label} shift': s for label, s in zip(b['labels'], shift)},
               **{f'{label} shift/sigma': s for label, s in
                  zip(b['labels'], shift / signed_sigma(shift, b['lo'], b['hi']))})
    if SPEC[row.fit].prior is None:
        rec['delta M_A [GeV]'] = rec['phys'][0] - to_physics(row.fit, b['best'])[0, 0]
    rec['bias at Q2ref'] = bias[IREF]
    rec['max |bias| Q2<=max'] = np.nanmax(np.abs(bias[IN_RANGE_Q2]))
    rec.update({'chain bias q16': q16, 'chain bias q50': q50, 'chain bias q84': q84})
    records.append(rec)

if not records:
    raise RuntimeError(f'no finished injection fits below {STUDY_ROOT}; run xml/injection_study/run_all.sh first')
RESULTS = pd.DataFrame(records).set_index(['fit', 'knob', 'sigma']).sort_index()
# linearity of the response: bias at 3 sigma over three times the bias at 1 sigma
lin = {}
for (fit, knob), part in RESULTS.groupby(level=[0, 1]):
    if {1.0, 3.0} <= set(part.index.get_level_values('sigma')):
        b1, b3 = part.loc[(fit, knob, 1.0), 'bias at Q2ref'], part.loc[(fit, knob, 3.0), 'bias at Q2ref']
        lin[(fit, knob, 3.0)] = b3 / (3 * b1) if b1 != 0 else np.nan
RESULTS['linearity'] = pd.Series(lin)
SHOW_COLS = [c for c in RESULTS.columns if c not in ('phys', 'bias')]

DISAGREE = 0.5
gap = (RESULTS['chain bias q50'] - RESULTS['bias at Q2ref']).abs()
suspect = RESULTS[gap > DISAGREE]
if len(suspect):
    print(f'\nminimizer and MCMC median differ by more than {DISAGREE} delta_F (check the log and posterior):')
    print(suspect[['chi2', 'bias at Q2ref', 'chain bias q16', 'chain bias q50', 'chain bias q84']]
          .to_string(float_format='{:+.2f}'.format))
else:
    print(f'\nminimizer and MCMC median agree within {DISAGREE} delta_F for every fit')

for fit in BASE_FITS:
    if fit not in BAND:
        continue
    part = RESULTS.loc[fit, SHOW_COLS].dropna(axis=1, how='all')
    order = part.xs(1.0, level='sigma')['bias at Q2ref'].abs().sort_values(ascending=False).index
    part = part.reindex(order, level='knob')
    print(f'\n{fit}: injections ranked by |F_A bias| at Q^2 = {Q2_REFERENCE:g} GeV^2 for the 1 sigma injection')
    fmt = {c: '{:+.3f}' for c in part.columns if 'shift' in c or 'delta' in c}
    fmt |= {c: '{:+.2f}' for c in part.columns if 'bias' in c} | {'linearity': '{:.2f}', 'chi2': '{:.2f}'}
    display(part.style.format(fmt, na_rep='').apply(
        lambda s: ['font-weight: bold' if abs(v) >= BIAS_NOTABLE else '' for v in s], subset=['bias at Q2ref']))

TABLE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS[SHOW_COLS].to_csv(TABLE_DIR / f'injection_results_{VARIANT}.csv', float_format='%.5g')
print(f'wrote {(TABLE_DIR / f"injection_results_{VARIANT}.csv").relative_to(REPO)}')

### Bias per knob

One panel per base fit, the knobs ranked by their largest 1 sigma bias in either fit. Bars are
the 1 sigma injections, hollow markers the 3 sigma injections divided by three (a marker off
its bar is a nonlinear response), the grey band is `|bias| < BIAS_NOTABLE`. Knobs above the
threshold in some fit keep one colour through the rest of the section. The quadrature sum over
knobs in each title is the total bias if every knob were mis-modelled by 1 sigma independently.

In [ ]:
FITS_DONE = [fit for fit in BASE_FITS if fit in BAND]
ONE = RESULTS.xs(1.0, level='sigma')
THREE = RESULTS.xs(3.0, level='sigma') if 3.0 in RESULTS.index.get_level_values('sigma') else None
RANK = ONE['bias at Q2ref'].abs().groupby(level='knob').max().sort_values(ascending=False)
KNOBS_RANKED = RANK.index.tolist()
NOTABLE = [k for k in KNOBS_RANKED if RANK[k] >= BIAS_NOTABLE]
PALETTE = ['#D55E00', '#0072B2', '#009E73', '#CC79A7', '#E69F00', '#56B4E9', '#000000', '#8B4513', '#6A3D9A', '#B2DF8A']
KNOB_COLOR = {k: PALETTE[i % len(PALETTE)] for i, k in enumerate(NOTABLE)}
GREY = '#c5c9cf'


def short(knob):
    return re.sub(r'_(UBGenie|SCC)$', '', knob)


def knob_color(knob):
    return KNOB_COLOR.get(knob, GREY)


Y_KNOB = np.arange(len(KNOBS_RANKED))[::-1]
fig, axes = plt.subplots(1, len(FITS_DONE), figsize=(4.0 * len(FITS_DONE) + 1.2, 0.27 * len(KNOBS_RANKED) + 1.6),
                         sharey=True, squeeze=False)
for ax, fit in zip(axes[0], FITS_DONE):
    b1 = ONE.loc[fit].reindex(KNOBS_RANKED)['bias at Q2ref']
    ax.axvspan(-BIAS_NOTABLE, BIAS_NOTABLE, color='#f1f3f5', zorder=0)
    ax.barh(Y_KNOB, b1, height=0.72, color=[knob_color(k) for k in KNOBS_RANKED], zorder=2)
    if THREE is not None and fit in THREE.index.get_level_values('fit'):
        b3 = THREE.loc[fit].reindex(KNOBS_RANKED)['bias at Q2ref'] / 3
        ax.scatter(b3, Y_KNOB, s=20, facecolor='white', edgecolor='#1b1f24', lw=0.9, zorder=3)
    ax.axvline(0, color='#1b1f24', lw=0.8)
    total = np.sqrt(np.nansum(b1 ** 2))
    ax.set_title(f'{fit}\n' + rf'$\delta_F = {100 * BAND[fit][2][IREF]:.1f}\%$, '
                 rf'quadrature sum of 1$\sigma$ biases = {total:.2f} $\delta_F$', fontsize=8.5)
    ax.set_xlabel(rf'$F_A$ bias at $Q^2 = {Q2_REFERENCE:g}$ GeV$^2$  [baseline $\delta_F$]', fontsize=9)
    lim = max(1.08 * np.nanmax(np.abs(b1)), 3 * BIAS_NOTABLE)
    ax.set_xlim(-lim, lim)
    ax.grid(axis='x', color='#e5e7eb', lw=0.5)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
axes[0][0].set_yticks(Y_KNOB, [short(k) for k in KNOBS_RANKED], fontsize=7.5)
handles = [plt.Rectangle((0, 0), 1, 1, color='#1b1f24'),
           plt.Line2D([], [], ls='', marker='o', mfc='white', mec='#1b1f24', ms=5),
           plt.Rectangle((0, 0), 1, 1, color='#f1f3f5')]
axes[0][-1].legend(handles, [r'1$\sigma$ injection', r'3$\sigma$ injection / 3', rf'$|$bias$| < {BIAS_NOTABLE:g}\,\delta_F$'],
                   fontsize=7, loc='lower right')
fig.suptitle(f'Injection study, {VARIANT} binning', fontsize=10)
fig.tight_layout()
fig.savefig(FIG_DIR / f'injection_bias_{VARIANT}.pdf')
plt.show()

### Bias against Q^2

The same 1 sigma biases as curves over `Q^2`, in units of the baseline `delta_F(Q^2)` so the two
fits share one axis and the band is the baseline uncertainty itself. F_A(0) is fixed in both
parametrisations, so the curves start at `Q^2 > 0`.

In [ ]:
def label_right(ax, entries, x, min_gap):
    '''Write labels at x for (y, text, colour) entries, pushed apart by at least min_gap.'''
    entries = sorted(entries)
    ys = [y for y, _, _ in entries]
    for i in range(1, len(ys)):
        ys[i] = max(ys[i], ys[i - 1] + min_gap)
    if ys:
        top = ax.get_ylim()[1] - 0.5 * min_gap
        for i in range(len(ys) - 1, -1, -1):
            if ys[i] > top:
                ys[i] = top
                top = ys[i] - min_gap
    for y, (y0, text, color) in zip(ys, entries):
        ax.annotate(text, (x, y0), xytext=(x + 0.06, y), textcoords='data', ha='left', va='center',
                    fontsize=6.5, color=color, annotation_clip=False,
                    arrowprops=dict(arrowstyle='-', color=color, lw=0.4, shrinkA=0, shrinkB=0))


Q2_PLOT_MIN = 0.05   # GeV^2: delta_F -> 0 at Q^2 = 0, where F_A is fixed
fig, axes = plt.subplots(1, len(FITS_DONE), figsize=(5.4 * len(FITS_DONE), 3.8), sharey=True, squeeze=False)
for ax, fit in zip(axes[0], FITS_DONE):
    ax.axhspan(-1, 1, color=FIT_COLOR[fit], alpha=0.12, lw=0, label=r'baseline $\pm\delta_F$')
    part = ONE.loc[fit]
    ends = []
    for knob in part.index:
        curve = np.where(Q2 >= Q2_PLOT_MIN, part.loc[knob, 'bias'], np.nan)
        if knob in KNOB_COLOR:
            ax.plot(Q2, curve, color=KNOB_COLOR[knob], lw=1.4, zorder=3)
            ends.append((curve[-1], short(knob), KNOB_COLOR[knob]))
        else:
            ax.plot(Q2, curve, color=GREY, lw=0.7, zorder=1)
    ax.axvline(Q2_REFERENCE, color='#8a93a0', lw=0.8, ls=':')
    ax.axhline(0, color='#1b1f24', lw=0.6)
    ax.set_xlim(0, Q2[-1] * 1.22)
    ax.set_xticks(np.arange(0, Q2[-1] + 0.01, 0.5))
    ax.set_xlabel(r'$Q^2$ [GeV$^2$]')
    ax.set_title(rf'{fit}: 1$\sigma$ injections', fontsize=9)
    ax.grid(color='#eceff1', lw=0.5)
    ax.set_axisbelow(True)
ylim = max(1.3, 1.1 * np.nanmax(np.abs(np.stack(ONE['bias'].values)[:, Q2 >= Q2_PLOT_MIN])))
axes[0][0].set_ylim(-ylim, ylim)
for ax, fit in zip(axes[0], FITS_DONE):
    part = ONE.loc[fit]
    ends = [(part.loc[k, 'bias'][-1], short(k), KNOB_COLOR[k]) for k in part.index if k in KNOB_COLOR]
    label_right(ax, ends, Q2[-1], 0.045 * 2 * ylim)
axes[0][0].set_ylabel(rf'$F_A$ bias  [baseline $\delta_F(Q^2)$]')
axes[0][0].legend(fontsize=7, loc='upper left')
fig.tight_layout()
fig.savefig(FIG_DIR / f'injection_fa_shift_{VARIANT}.pdf')
plt.show()

### Where the fits move in parameter space

The fitted axial parameters themselves. For the dipole fit the shift of M_A in GeV against the
baseline 68% posterior interval; for the z-expansion fit the shift in the (PCA1, PCA2) pull
plane against the baseline posterior's 68% and 95% ellipses (from the chain covariance). Filled
points are the 1 sigma injections, hollow points the 3 sigma injections at full scale (not divided by three), joined to
their 1 sigma point: three times as far from the origin for a linear response.

In [ ]:
from matplotlib.patches import Ellipse


def cov_ellipse(ax, cov, center, dchi2, **kw):
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    w, h = 2 * np.sqrt(dchi2 * vals)
    ax.add_patch(Ellipse(center, w, h, angle=angle, **kw))


fig, axes = plt.subplots(1, len(FITS_DONE), figsize=(4.8 * len(FITS_DONE), 4.2), squeeze=False)
for ax, fit in zip(axes[0], FITS_DONE):
    b = BASELINE[fit]
    labels = b['labels']
    part1 = ONE.loc[fit].reindex(KNOBS_RANKED)
    part3 = THREE.loc[fit].reindex(KNOBS_RANKED) if THREE is not None and fit in THREE.index.get_level_values('fit') else None
    if len(labels) == 1:
        # dipole M_A: one dimension, knobs along y
        ma_chain = to_physics(fit, b['chain'])[:, 0]
        ma_best = to_physics(fit, b['best'])[0, 0]
        q16, q84 = np.quantile(ma_chain, [0.16, 0.84]) - ma_best
        ax.axvspan(q16, q84, color=FIT_COLOR[fit], alpha=0.15, lw=0, label='baseline 68% posterior')
        d1 = part1['delta M_A [GeV]']
        ax.scatter(d1, Y_KNOB, s=26, color=[knob_color(k) for k in KNOBS_RANKED], zorder=3, label=r'1$\sigma$ injection')
        if part3 is not None:
            d3 = part3['delta M_A [GeV]']
            ax.hlines(Y_KNOB, d1, d3, color='#9aa0a6', lw=0.6, zorder=2)
            ax.scatter(d3, Y_KNOB, s=20, facecolor='white', edgecolor='#1b1f24', lw=0.9, zorder=3, label=r'3$\sigma$ injection')
        ax.axvline(0, color='#1b1f24', lw=0.8)
        ax.set_yticks(Y_KNOB, [short(k) for k in KNOBS_RANKED], fontsize=7)
        ax.set_xlabel(r'$\Delta M_A$ [GeV]')
        ax.legend(fontsize=7, loc='lower left')
        ax.grid(axis='x', color='#e5e7eb', lw=0.5)
    else:
        # z expansion: the first two pulls
        chain = b['chain'][:, :2] - b['best'][:2]
        cov = np.cov(chain, rowvar=False)
        for dchi2, alpha in ((5.99, 0.10), (2.30, 0.22)):
            cov_ellipse(ax, cov, (0, 0), dchi2, facecolor=FIT_COLOR[fit], alpha=alpha, lw=0)
        c1 = part1[[f'{labels[0]} shift', f'{labels[1]} shift']].to_numpy()
        if part3 is not None:
            c3 = part3[[f'{labels[0]} shift', f'{labels[1]} shift']].to_numpy()
            for p1, p3 in zip(c1, c3):
                if np.all(np.isfinite(p3)):
                    ax.plot([p1[0], p3[0]], [p1[1], p3[1]], color='#9aa0a6', lw=0.6, zorder=2)
            ax.scatter(c3[:, 0], c3[:, 1], s=20, facecolor='white', edgecolor='#1b1f24', lw=0.9, zorder=3,
                       label=r'3$\sigma$ injection')
        ax.scatter(c1[:, 0], c1[:, 1], s=26, color=[knob_color(k) for k in KNOBS_RANKED], zorder=4,
                   label=r'1$\sigma$ injection')
        for k, p in zip(KNOBS_RANKED, c3 if part3 is not None else c1):
            if k in NOTABLE[:4] and np.all(np.isfinite(p)):
                ax.annotate(short(k), p, xytext=(4, 3), textcoords='offset points', fontsize=6.5, color=KNOB_COLOR[k])
        ax.axhline(0, color='#1b1f24', lw=0.6)
        ax.axvline(0, color='#1b1f24', lw=0.6)
        # inset: the 1 sigma injections, which sit deep inside the posterior
        rz = 1.25 * np.nanmax(np.abs(c1))
        ins = ax.inset_axes([0.04, 0.04, 0.40, 0.40])
        for dchi2, alpha in ((5.99, 0.10), (2.30, 0.22)):
            cov_ellipse(ins, cov, (0, 0), dchi2, facecolor=FIT_COLOR[fit], alpha=alpha, lw=0)
        ins.scatter(c1[:, 0], c1[:, 1], s=14, color=[knob_color(k) for k in KNOBS_RANKED], zorder=4)
        for k, p in zip(KNOBS_RANKED, c1):
            if k in NOTABLE[:4]:
                ins.annotate(short(k), p, xytext=(3, 2), textcoords='offset points', fontsize=5.5, color=KNOB_COLOR[k])
        ins.axhline(0, color='#1b1f24', lw=0.5)
        ins.axvline(0, color='#1b1f24', lw=0.5)
        ins.set_xlim(-rz, rz)
        ins.set_ylim(-rz, rz)
        ins.set_aspect('equal')
        ins.tick_params(labelsize=5.5, length=2)
        ins.set_title(r'1$\sigma$ injections', fontsize=6, pad=2)
        ax.indicate_inset_zoom(ins, edgecolor='#8a93a0', lw=0.6)
        r = 1.15 * max(np.nanmax(np.abs(c1)), np.nanmax(np.abs(c3)) if part3 is not None else 0, np.sqrt(2.30 * cov.max()))
        ax.set_xlim(-r, r)
        ax.set_ylim(-r, r)
        ax.set_aspect('equal')
        ax.set_xlabel(f'{labels[0]} shift [pull]')
        ax.set_ylabel(f'{labels[1]} shift [pull]')
        handles = [plt.Rectangle((0, 0), 1, 1, color=FIT_COLOR[fit], alpha=0.3)]
        h, l = ax.get_legend_handles_labels()
        ax.legend(handles + h, ['baseline 68% / 95% posterior'] + l, fontsize=7, loc='upper right')
        ax.set_ylim(-r, r)
        ax.grid(color='#eceff1', lw=0.5)
    ax.set_axisbelow(True)
    ax.set_title(fit, fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / f'injection_parameters_{VARIANT}.pdf')
plt.show()

### Response to the injected shift, minimizer against MCMC

The F_A bias at `Q2_REFERENCE` against the injected shift for every knob with both injections.
Lines join the minimizer's best fits (a straight line through the origin is a linear response);
the markers with error bars are the median and 16-84% interval of each injected fit's own chain,
drawn for the highlighted knobs. A marker far from its line is the disagreement flagged in the
table above.

In [ ]:
fig, axes = plt.subplots(1, len(FITS_DONE), figsize=(4.6 * len(FITS_DONE), 3.8), squeeze=False)
for ax, fit in zip(axes[0], FITS_DONE):
    part = RESULTS.loc[fit]
    for knob in KNOBS_RANKED:
        if knob not in part.index.get_level_values('knob'):
            continue
        p = part.loc[knob].sort_index()
        s = np.r_[0.0, p.index.to_numpy()]
        y = np.r_[0.0, p['bias at Q2ref'].to_numpy()]
        if knob in KNOB_COLOR:
            ax.plot(s, y, '-', color=KNOB_COLOR[knob], lw=1.3, zorder=3, label=short(knob))
            q16, q50, q84 = (p[c].to_numpy() for c in ('chain bias q16', 'chain bias q50', 'chain bias q84'))
            ax.errorbar(p.index.to_numpy(), q50, yerr=[q50 - q16, q84 - q50],
                        fmt='o', ms=3.5, color=KNOB_COLOR[knob], capsize=2, lw=0.8, zorder=4)
        else:
            ax.plot(s, y, '-', color=GREY, lw=0.7, zorder=1)
    ax.axhline(0, color='#1b1f24', lw=0.6)
    ax.axhspan(-1, 1, color=FIT_COLOR[fit], alpha=0.08, lw=0)
    ax.set_xticks([0, 1, 3])
    ax.set_xlim(-0.15, 3.4)
    ax.set_xlabel(r'injected shift of the knob [$\sigma$]')
    ax.set_title(fit, fontsize=9)
    ax.grid(color='#eceff1', lw=0.5)
    ax.set_axisbelow(True)
axes[0][0].set_ylabel(rf'$F_A$ bias at $Q^2 = {Q2_REFERENCE:g}$ GeV$^2$  [baseline $\delta_F$]')
handles = [plt.Line2D([], [], color=KNOB_COLOR[k], lw=1.3) for k in NOTABLE]
handles += [plt.Line2D([], [], color='#1b1f24', lw=1.3), plt.Line2D([], [], ls='', marker='o', color='#1b1f24', ms=3.5)]
fig.legend(handles, [short(k) for k in NOTABLE] + ['best fit', 'chain median, 16-84%'], fontsize=7,
           ncol=min(len(handles), 5), loc='lower center', bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1))
fig.savefig(FIG_DIR / f'injection_response_{VARIANT}.pdf')
plt.show()